# 02 - Chunk, Embed, and Store Resumes

This notebook loads the parsed resume documents, splits them into overlapping chunks, creates Gemini embeddings, and stores them in a local Chroma vector database. It works for one resume or many PDFs in `data/`.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import re

from dotenv import load_dotenv


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    if (current / "data" / "resume_documents.json").exists():
        return current
    if (current.parent / "data" / "resume_documents.json").exists():
        return current.parent
    raise FileNotFoundError("Run 01_parse_resume.ipynb first to create data/resume_documents.json.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PARSED_DOCS_PATH = DATA_DIR / "resume_documents.json"
CHUNKS_PATH = DATA_DIR / "resume_chunks.json"
CHROMA_DIR = PROJECT_ROOT / "chroma_db"

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise EnvironmentError("GOOGLE_API_KEY is missing. Add it to your .env file.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Chroma DB:    {CHROMA_DIR}")

Project root: /Users/atharv/projects/attem
Chroma DB:    /Users/atharv/projects/attem/chroma_db


In [2]:
documents = json.loads(PARSED_DOCS_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(documents)} parsed document page(s).")


def split_text(text: str, chunk_size: int = 800, overlap: int = 150) -> list[dict]:
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks = []
    current = ""

    for paragraph in paragraphs:
        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) <= chunk_size:
            current = candidate
            continue

        if current:
            chunks.append(current)
        current = paragraph

        while len(current) > chunk_size:
            chunks.append(current[:chunk_size].strip())
            current = current[chunk_size - overlap :].strip()

    if current:
        chunks.append(current)

    with_offsets = []
    search_from = 0
    for chunk in chunks:
        start = text.find(chunk[:80], search_from)
        if start == -1:
            start = search_from
        end = start + len(chunk)
        with_offsets.append({"text": chunk, "char_start": start, "char_end": end})
        search_from = max(end - overlap, 0)
    return with_offsets


chunks = []
for doc in documents:
    page_chunks = split_text(doc["text"], chunk_size=800, overlap=150)
    for local_index, chunk in enumerate(page_chunks):
        metadata = dict(doc["metadata"])
        metadata.update(
            {
                "chunk_index": len(chunks),
                "page_chunk_index": local_index,
                "char_start": chunk["char_start"],
                "char_end": chunk["char_end"],
            }
        )
        chunk_id = hashlib.sha1(f"{metadata['source']}:{metadata['page']}:{local_index}:{chunk['text']}".encode()).hexdigest()
        chunks.append({"id": chunk_id, "text": chunk["text"], "metadata": metadata})

CHUNKS_PATH.write_text(json.dumps(chunks, indent=2), encoding="utf-8")
print(f"Created {len(chunks)} chunk(s).")
print(f"Saved chunks to {CHUNKS_PATH}")
chunks[:2]

Loaded 1 parsed document page(s).
Created 6 chunk(s).
Saved chunks to /Users/atharv/projects/attem/data/resume_chunks.json


[{'id': 'cba09a0152c6ce70215744b467bf1cad424e6122',
  'text': 'Atharv Tembhurnikar\nCollege Park, MD, USA\n\x83 +1 240-886-6670\n# attem@umd.edu\nï LinkedIn\n§ GitHub \x80 Portfolio\nBlogs IEEE Profile\nExperience\nTeaching Assistant (Agentic AI) – University of Maryland (UMD)\nAug 2026 – Present\nMachine Learning Research Assistant – University of Maryland (UMD)\nDec 2025 – Jul 2026\n• Developed ML-based pipelines for cell-wise annotation and intensity tracking on artificial-nose sensor data, enabling\nstructured capture of cellular responses and temporal patterns over time.\n• Engineered interpretable feature representations (response amplitude, decay dynamics, temporal slopes) to quantify cell\nbehavior, supporting data-driven validation of experimental hypotheses and robust pattern classification.\nData Science Intern – Hackveda Private Limited (Pune, India)\nJan 2025 –',
  'metadata': {'source': 'resume.pdf',
   'page': 1,
   'parser': 'pymupdf',
   'chunk_index': 0,
   'page_chun

In [3]:
from google import genai
from google.genai import types
from tqdm.auto import tqdm

EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSIONS = 768

genai_client = genai.Client(api_key=api_key)


def embed_texts(texts: list[str], task_type: str, batch_size: int = 16) -> list[list[float]]:
    embeddings = []
    for start in tqdm(range(0, len(texts), batch_size), desc=f"Embedding {task_type.lower()}"):
        batch = texts[start : start + batch_size]
        result = genai_client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(
                task_type=task_type,
                output_dimensionality=EMBEDDING_DIMENSIONS,
            ),
        )
        embeddings.extend([embedding.values for embedding in result.embeddings])
    return embeddings


chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = embed_texts(chunk_texts, task_type="RETRIEVAL_DOCUMENT")
print(f"Created {len(chunk_embeddings)} embedding(s), dimension {len(chunk_embeddings[0])}.")

Embedding retrieval_document:   0%|          | 0/1 [00:00<?, ?it/s]

Created 6 embedding(s), dimension 768.


In [4]:
import chromadb

COLLECTION_NAME = "resume_rag"

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    ids=[chunk["id"] for chunk in chunks],
    documents=chunk_texts,
    embeddings=chunk_embeddings,
    metadatas=[chunk["metadata"] for chunk in chunks],
)

print(f"Stored {collection.count()} chunk(s) in Chroma collection '{COLLECTION_NAME}'.")

Stored 6 chunk(s) in Chroma collection 'resume_rag'.


In [5]:
test_query = "What are my strongest technical skills?"
query_embedding = embed_texts([test_query], task_type="RETRIEVAL_QUERY", batch_size=1)[0]

results = collection.query(query_embeddings=[query_embedding], n_results=3)

for rank, (text, metadata, distance) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1,
):
    print(f"\nResult {rank} | {metadata['source']} | page {metadata['page']} | distance {distance:.4f}\n{text[:700]}")

Embedding retrieval_query:   0%|          | 0/1 [00:00<?, ?it/s]


Result 1 | resume.pdf | page 1 | distance 0.3312
ta Science Tools: Python, Java, MySQL, PostgreSQL, MongoDB, Amazon Web Services,
NumPy, Pandas, Scikit-learn, PyTorch, Langchain, Langraph, Power BI, Tableau, Git
Domains: Machine Learning, Deep Learning, Natural Language Processing, Gen AI, Agentic AI, Business Intelligence
Other Skills & Methodologies: Research, Mathematics and Statistics, Data Analytics, Agile Methodology
Projects and Research Publications
DCFare | Python, XGBoost, Neural Networks
Aug 2025 - Dec 2025
∗Built an end-to-end predictive pipeline on 2.5M+ DC taxicab trips; performed systematic feature engineering and model
selection, comparing ensemble and neural approaches to quantify performance–data scale trade-offs.
∗Achiev

Result 2 | resume.pdf | page 1 | distance 0.3611
uantify performance–data scale trade-offs.
∗Achieved R² of 0.96 with an MLP neural network; conducted ethics-aware analysis addressing privacy, fairness, and
responsible use of mobility data.
EcoInte